In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np

In [3]:
# load dataset
df=pd.read_csv('C:\Shopper Spectrum\online_retail.csv')
df.head()

<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
C:\Users\PRATHAP\AppData\Local\Temp\ipykernel_19264\2904812642.py:2: SyntaxWarning: invalid escape sequence '\S'
  df=pd.read_csv('C:\Shopper Spectrum\online_retail.csv')


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSpend
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2022-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2022-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2022-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2022-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2022-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [5]:
#change date column to right datetime format: 
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

In [6]:
#1 Run the K-Means Clustering Model

snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, 
    'InvoiceNo': 'nunique',                                  
    'TotalSpend': 'sum'                                      
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']

# 2. Standardize the data (scales all columns so large numbers don't confuse the model)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# 3. Run K-Means with 4 clusters (as decided by your Elbow Curve)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

print("K-Means Clustering successfully completed!")

K-Means Clustering successfully completed!


In [7]:
#2. Name Your Segments based on the project requirements

# Cluster 2 = VIP, Cluster 0 = Regular, Cluster 3 = Occasional, Cluster 1 = At-Risk
cluster_names = {
    2: 'High-Value / VIP',
    0: 'Regular Shopper',
    3: 'Occasional Shopper',
    1: 'At-Risk Customer'
}


rfm['Segment Label'] = rfm['Cluster'].map(cluster_names)


display(rfm[['Recency', 'Frequency', 'Monetary', 'Segment Label']].head())

,Recency,Frequency,Monetary,Segment Label
CustomerID,,,,
12346.0,326,1,77183.60,Occasional Shopper
12347.0,2,7,4310.00,Regular Shopper
12348.0,75,4,1797.24,Regular Shopper
12349.0,19,1,1757.55,Regular Shopper
12350.0,310,1,334.40,At-Risk Customer


In [8]:
#3. Save Your Model for your Streamlit App:
import pickle

import os
if not os.path.exists('models'):
    os.makedirs('models')

# Save the trained K-Means model
with open('models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

# Save the StandardScaler 
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Your Machine Learning models have been successfully saved inside the 'models/' folder!")

Your Machine Learning models have been successfully saved inside the 'models/' folder!


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

print("Preparing data for Recommendation Engine...")

# 1. To keep your computer running fast, let's look at the top 500 most popular items
top_items = df['Description'].value_counts().head(500).index
df_subset = df[df['Description'].isin(top_items)]

# 2. Create a Pivot Table (Rows = Products, Columns = CustomerIDs, Values = Total Quantity bought)

pivot_matrix = df_subset.pivot_table(
    index='Description', 
    columns='CustomerID', 
    values='Quantity', 
    aggfunc='sum'
).fillna(0)

# 3. Calculate Cosine Similarity (finds products with matching purchase patterns)
item_similarity = cosine_similarity(pivot_matrix)

# 4. Turn it into a clean DataFrame for easy searching
item_sim_df = pd.DataFrame(item_similarity, index=pivot_matrix.index, columns=pivot_matrix.index)

print("Recommendation matrix successfully built!")

Preparing data for Recommendation Engine...
Recommendation matrix successfully built!


In [10]:
def get_recommendations(product_name, similarity_data=item_sim_df):
    if product_name not in similarity_data.index:
        return f"Product '{product_name}' not found in the dataset index. Please check spelling!"
    
    # Get the similarities for this product, sort them from highest to lowest, and grab top 6
    similar_products = similarity_data[product_name].sort_values(ascending=False).iloc[1:6]
    
    return pd.DataFrame(similar_products).rename(columns={product_name: 'Similarity Score'})

# Let's test it with one of your best sellers! 
# (Make sure this matches an exact name from your top products list, like 'WHITE HANGING HEART T-LIGHT HOLDER')
test_product = 'WHITE HANGING HEART T-LIGHT HOLDER'
print(f"Top 5 Recommendations for: {test_product}")
display(get_recommendations(test_product))

Top 5 Recommendations for: WHITE HANGING HEART T-LIGHT HOLDER


,Similarity Score
Description,
GIN + TONIC DIET METAL SIGN,0.750192
RED HANGING HEART T-LIGHT HOLDER,0.658714
WASHROOM METAL SIGN,0.643520
LAUNDRY 15C METAL SIGN,0.642200
HEART OF WICKER LARGE,0.628669


In [11]:
# Save the similarity matrix dataframe to your models folder
with open('models/item_similarity.pkl', 'wb') as f:
    pickle.dump(item_sim_df, f)

print("Recommendation matrix saved successfully to 'models/item_similarity.pkl'!")

Recommendation matrix saved successfully to 'models/item_similarity.pkl'!
